# 01 - Clean Crop Production (Silver)

From `bronze.crop_production_raw`:
1. Strip string columns; normalize `crop_group`
2. Aggregate duplicate business keys (`province + crop + year + quarter`) -> sum production
3. Round numerics; ensure boolean types
4. Recompute lags, rolling anomalies, YoY growth, climate stress **after** aggregation
5. Write `silver.crop_production_clean` and validate

## 1. Configuration

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

from configs.project_config import (
    BRONZE_TABLES, 
    SILVER_TABLES, 
    BUSINESS_KEY_COLS, 
    VALID_QUARTERS
)

bronze_table = BRONZE_TABLES['crop_production_raw']
silver_table = SILVER_TABLES['crop_production_clean']

print(f'Source: {bronze_table}')
print(f'Target: {silver_table}')

## 2. Load Bronze

In [0]:
raw = spark.table(bronze_table)
print(f'Bronze rows: {raw.count()}')

## 3. Strip strings + normalize categoricals

In [0]:
df = (
    raw
    .withColumn('province', F.trim(F.col('province')))
    .withColumn('crop', F.trim(F.col('crop')))
    .withColumn('quarter', F.trim(F.col('quarter')))
    .withColumn('crop_group', F.upper(F.trim(F.col('crop_group'))))
    .withColumn('risk_label', F.trim(F.col('risk_label')))
)

## 4. Aggregate duplicate business keys

Sum production. For weather / climate columns take first (identical within key).
Drop pre-aggregation lags / growth / stress - they will be recomputed.

In [0]:
keep_first = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'precipitation_sum', 'rain_sum', 'precipitation_hours',
    'sunshine_duration', 'shortwave_radiation_sum',
    'reference_evapotranspiration_mm',
    'wind_speed_10m_max', 'wind_gusts_10m_max',
    'rain_normal", "temp_normal',
    'rainfall_deviation_pct', 'temperature_anomaly_c', 'oni_index',
    'quarter_num', 'time_index',
    'enso_phase_la_nina', 'enso_phase_neutral',
    'crop_group', 'risk_label'
]

agg_exprs = [F.sum('production').alias('production')]
agg_exprs += [F.first(c, ignorenulls=True).alias(c) for c in keep_first if c in df.columns]
agg_exprs += [
    F.max('_ingested_at').alias('_ingested_at'),
    F.first('_source_file').alias('_source_file')
]

df = df.groupBy(*BUSINESS_KEY_COLS).agg(*agg_exprs)

## 5. Round numerics + cast types

In [0]:
round_2 = [
    'temperature_2m_mean', 'temperature_2m_max', 'temperature_2m_min',
    'precipitation_sum', 'rain_sum',
    'shortwave_radiation_sum', 'reference_evapotranspiration_mm',
    'wind_speed_10m_max', 'wind_gusts_10m_max',
    'rain_normal', 'temp_normal', 'rainfall_deviation_pct'
]
round_3 = [
    'temperature_anomaly_c', 'oni_index', 'production'
]
round_0 = ['sunshine_duration']

for c in round_2:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 2))
for c in round_3:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 3))
for c in round_0:
    if c in df.columns:
        df = df.withColumn(c, F.round(F.col(c), 0))

if 'precipitation_hours' in df.columns:
    df = df.withColumn('precipitation_hours', F.col('precipitation_hours').cast('int'))

df = (
    df
    .withColumn('enso_phase_la_nina', F.col('enso_phase_la_nina').cast('boolean'))
    .withColumn('enso_phase_neutral', F.col('enso_phase_neutral').cast('boolean'))
)

## 6.